# IK end-effector control

This notebook replaces the terminal UI from `../../WBC/ik_pose_cli_v3.py` with a Jupyter widget panel. It sends small Cartesian end-effector increments through `ArmSdk.ik_move_EE()` and clamps joint changes per command with `max_dq`.


In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Import `ArmSdk`, widgets, and numeric helpers.


In [ ]:
import json
import time
import numpy as np
import ipywidgets as widgets
from IPython.display import display

from arm_sdk import ArmSdk

from sdk_client import Robot
from inspire_sdk import close_hand as inspire_close_hand, open_hand as inspire_open_hand


Create the IK controller and sync it to the current measured upper-body state.


In [ ]:
ik = ArmSdk(iface=IFACE, domain_id=DOMAIN_ID)
ik.resync()
print("ArmSdk IK controller synced to current state.")


Helpers for EE pose display and single-increment commands.


In [ ]:
DOF_INDEX = {"x": 0, "y": 1, "z": 2, "roll": 3, "pitch": 4, "yaw": 5}


def pose_summary(arm):
    # TODO: Read the current end-effector pose and format it for the notebook status view.
    raise NotImplementedError("Participant exercise: complete pose_summary.")


def ramped_ik_move_EE(
    pose_increment,
    *,
    arm="right",
    position_only=False,
    selected_axis=None,
    max_dq=0.04,
    cart_speed_m_s=0.04,
    rot_speed_rad_s=0.25,
    rate_hz=20.0,
):
    # TODO: Break a Cartesian pose increment into safe IK steps and send them at a fixed rate.
    raise NotImplementedError("Participant exercise: complete ramped_ik_move_EE.")


def apply_increment(
    arm,
    dof,
    signed_step,
    position_only=False,
    max_dq=0.04,
    cart_speed_m_s=0.04,
    rot_speed_rad_s=0.25,
    rate_hz=20.0,
):
    # TODO: Build the 6D pose increment from UI values, run the ramped IK move, and refresh status.
    raise NotImplementedError("Participant exercise: complete apply_increment.")


robot_control = None


def get_robot_control():
    # TODO: Lazily create and cache the high-level Robot control client.
    raise NotImplementedError("Participant exercise: complete get_robot_control.")


def normalize_hand_selection(hand):
    # TODO: Map the UI hand selector into the sides expected by the hand SDK.
    raise NotImplementedError("Participant exercise: complete normalize_hand_selection.")


def hand_action(hand_type, action, hand="both", hold_s=0.6, ramp_s=0.4):
    # TODO: Dispatch open/close commands to the selected hand implementation and return a status dict.
    raise NotImplementedError("Participant exercise: complete hand_action.")


Run the panel. Use small translation and rotation steps, and resync after physical contact or manual repositioning.


In [ ]:
arm = widgets.ToggleButtons(options=["left", "right", "both"], value="right", description="Arm")
dof = widgets.Dropdown(options=list(DOF_INDEX), value="x", description="DOF")
translation_step = widgets.FloatSlider(value=0.02, min=0.002, max=0.08, step=0.002, description="m step")
rotation_step = widgets.FloatSlider(value=0.08, min=0.01, max=0.30, step=0.01, description="rad step")
max_dq = widgets.FloatSlider(value=0.04, min=0.01, max=0.12, step=0.005, description="max dq")
cart_speed = widgets.FloatSlider(value=0.04, min=0.005, max=0.12, step=0.005, description="m/s")
rot_speed = widgets.FloatSlider(value=0.25, min=0.05, max=0.8, step=0.05, description="rad/s")
ramp_rate = widgets.FloatSlider(value=20.0, min=5.0, max=50.0, step=1.0, description="Hz")
position_only = widgets.Checkbox(value=False, description="free orientation for xyz")
hand_type = widgets.Dropdown(options=["dex3", "inspire"], value="dex3", description="Hand type")
hand_side = widgets.Dropdown(options=["both", "left", "right"], value="both", description="Hand")
minus = widgets.Button(description="- Step")
plus = widgets.Button(description="+ Step", button_style="success")
resync = widgets.Button(description="Resync", button_style="info")
release = widgets.Button(description="Release Arms", button_style="warning")
reengage = widgets.Button(description="Reengage Arms", button_style="success")
open_hand_btn = widgets.Button(description="Open Hand")
close_hand_btn = widgets.Button(description="Close Hand")
status = widgets.Textarea(layout=widgets.Layout(width="100%", height="300px"), disabled=True)


def current_step():
    # TODO: Return the signed step size selected in the UI.
    raise NotImplementedError("Participant exercise: complete current_step.")


def refresh(extra=None):
    # TODO: Read the latest robot/IK state and update the displayed status.
    raise NotImplementedError("Participant exercise: complete refresh.")


def move(sign):
    # TODO: Apply the selected axis/direction increment from the UI controls.
    raise NotImplementedError("Participant exercise: complete move.")


def on_release(_):
    # TODO: Complete the implementation for on_release using the surrounding notebook context.
    raise NotImplementedError("Participant exercise: complete on_release.")


def on_reengage(_):
    # TODO: Complete the implementation for on_reengage using the surrounding notebook context.
    raise NotImplementedError("Participant exercise: complete on_reengage.")


def on_hand(action):
    # TODO: Complete the implementation for on_hand using the surrounding notebook context.
    raise NotImplementedError("Participant exercise: complete on_hand.")

minus.on_click(lambda _: move(-1.0))
plus.on_click(lambda _: move(+1.0))
resync.on_click(lambda _: (ik.resync(), refresh("resynced")))
release.on_click(on_release)
reengage.on_click(on_reengage)
open_hand_btn.on_click(lambda _: on_hand("open"))
close_hand_btn.on_click(lambda _: on_hand("close"))
refresh("ready")
display(widgets.VBox([
    widgets.HBox([arm, dof, position_only]),
    widgets.HBox([translation_step, rotation_step, max_dq]),
    widgets.HBox([cart_speed, rot_speed, ramp_rate]),
    widgets.HBox([minus, plus, resync, release, reengage]),
    widgets.HBox([hand_type, hand_side, open_hand_btn, close_hand_btn]),
    status,
]))
